# MixMIL with AnnData: continuous phenotype prediction

MixMIL supports continuous outcomes through its Gaussian likelihood. The model predicts donor-level means and learns an observation-noise scale alongside the attention and fixed-effect parameters.

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import torch

from mixmil import MixMIL
from mixmil.anndata import add_bag_predictions_to_anndata, prepare_anndata

rng = np.random.default_rng(19)
n_bags, cells_per_bag, n_features = 64, 22, 7
embedding = rng.normal(size=(n_bags * cells_per_bag, n_features)).astype('float32')
bag_embedding = embedding.reshape(n_bags, cells_per_bag, n_features).mean(axis=1)
phenotype = (1.4 * bag_embedding[:, 0] - 0.8 * bag_embedding[:, 2] + rng.normal(scale=0.25, size=n_bags)).astype('float32')
obs = pd.DataFrame({
    'donor': np.repeat([f'donor_{i:03d}' for i in range(n_bags)], cells_per_bag),
    'split': np.repeat(np.where(np.arange(n_bags) < 48, 'train', 'test'), cells_per_bag),
    'continuous_phenotype': np.repeat(phenotype, cells_per_bag),
    'age': np.repeat(rng.integers(20, 80, size=n_bags), cells_per_bag),
})
adata = ad.AnnData(X=np.zeros((len(obs), 1), dtype='float32'), obs=obs)
adata.obsm['X_mixmil'] = embedding
adata

In [ ]:
prepared = prepare_anndata(
    adata,
    bag_key='donor',
    target_key='continuous_phenotype',
    feature_key='X_mixmil',
    fixed_effects=['age'],
    scale_fixed_effects=['age'],
    split_key='split',
)

model = MixMIL.init_with_mean_model(
    prepared.train.Xs, prepared.train.F, prepared.train.Y,
    likelihood='gaussian', mean_field=True,
)
model.fit(prepared.train.Xs, prepared.train.F, prepared.train.Y, n_epochs=200, batch_size=16, verbose=False)
prediction = model.predict(prepared.test.Xs, F=prepared.test.F).squeeze(1)
truth = prepared.test.Y.squeeze(1)
mse = torch.mean((prediction - truth) ** 2)
print('test MSE:', float(mse))
print('learned observation scale:', float(torch.exp(model.log_sigma_y[0, 0])))

Continuous predictions are stored as donor-level results in `.uns`. They are not copied into every cell because the phenotype is defined at the donor/bag level.

In [ ]:
result = add_bag_predictions_to_anndata(
    adata, prepared.test.bag_ids, prediction[:, None],
    key='mixmil_continuous',
)
result.uns['mixmil_continuous']['target'] = 'continuous_phenotype'
result.uns['mixmil_continuous']